# Gamerdust - Cuaderno tecnico y funcional

Este cuaderno explica el estado actual de la aplicacion Gamerdust, creada con Laravel, Inertia y React dentro de un monorepo (`apps/api`, `apps/web`, `apps/narrative-engine`). Esta pensado para entender que hace la app, como se conectan sus piezas, que tablas existen, cual es el flujo del usuario y si actualmente usa IA real o solo logica local.

**Fecha de revision:** 2026-10-06.

**Resumen rapido:** Gamerdust es una herramienta para construir y revisar narrativa de videojuegos. Permite crear proyectos narrativos (con genero, idioma y favoritos), definir restricciones creativas, registrar lore, personajes, escenas y nodos de dialogo, y despues revisar validaciones narrativas, analisis emocional simulado y estimaciones de costo narrativo.

**Cambios respecto a la version anterior del cuaderno (2026-06-08):**

- El proyecto se reorganizo como monorepo: Laravel vive ahora en `apps/api` y el frontend esta organizado por funcionalidades (`features/`).
- Nueva portada publica en `/` (landing) con el nuevo diseno oscuro/verde.
- Listado de proyectos con busqueda, filtro de favoritos y vista en cuadricula/lista; creacion de proyecto con genero e idioma.
- Nuevas columnas `language` y `favorite` en `projects` y nueva ruta para marcar favoritos.


## 0. Estructura del repositorio

```
gamerdustv1/
├── apps/
│   ├── api/                 Laravel 13 (backend, MySQL) + frontend React con Inertia
│   ├── web/                 Reservado para la interfaz como SPA independiente (solo README)
│   └── narrative-engine/    Motor de analisis en Python (esqueleto vacio, pendiente)
├── docs/                    requirements, backlog, architecture (este cuaderno), mockups, testing
├── tests/e2e/               Pruebas de recorridos completos (pendiente)
├── .vscode/                 extensions.json, launch.json, tasks.json
├── compose.yaml             Servicios de desarrollo local (MySQL)
├── .editorconfig, .gitignore
└── README.md                Instalacion y ejecucion
```

Todas las rutas de este cuaderno (`app/`, `routes/`, `resources/js/`, `database/`) son relativas a `apps/api`.

El C4 contempla una API NestJS con PostgreSQL; hoy el backend real es Laravel con MySQL. La carpeta `apps/api` conserva Laravel hasta que se decida esa migracion.


## 1. Stack y arquitectura general

La aplicacion combina backend Laravel con frontend React usando Inertia.

| Capa | Tecnologia | Archivos principales | Funcion |
|---|---|---|---|
| Backend | Laravel 13, PHP 8.4+ | `app/`, `routes/web.php`, `database/` | Rutas, controladores, modelos, autorizacion, validacion, persistencia |
| Frontend | React 19 + TypeScript | `resources/js/` | Paginas, formularios y paneles de la interfaz |
| Puente Laravel/React | Inertia | `app/Http/Middleware/HandleInertiaRequests.php`, `resources/js/app.tsx`, `resources/js/app/resolve-page.ts` | Laravel renderiza paginas React sin API REST separada |
| Build frontend | Vite + Tailwind CSS 4 + lucide-react | `vite.config.js`, `resources/css/app.css` | Compilacion, estilos y iconos |
| Persistencia | MySQL (configurado en `.env`) | `config/database.php`, migraciones | Tablas de usuarios y dominio narrativo |
| Motor de analisis | Python (pendiente) | `apps/narrative-engine` | Esqueleto para procesamiento de texto, VAD, coherencia e IA |

No hay una API JSON separada en este estado. Las acciones de usuario se envian con formularios Inertia a rutas web de Laravel. Laravel valida, guarda, redirige y vuelve a renderizar la pagina React correspondiente con props actualizadas.

**Diseno visual:** tema oscuro con color primario verde (`#c2e88d`), definido en `resources/css/app.css` (variables `--background`, `--foreground`, `--border`, `--primary`) con las fuentes DM Sans, Space Grotesk, IBM Plex Mono, Outfit y Pixelify Sans. Las paginas internas de lore, personajes, escenas y dialogos aun conservan el estilo anterior zinc/ambar.


In [ ]:
# Celda opcional: inventario rapido del proyecto Laravel (apps/api).
from pathlib import Path

def find_api_root(start=None):
    """Sube desde el directorio actual hasta encontrar apps/api (o el propio proyecto Laravel)."""
    here = (start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / 'apps' / 'api' / 'artisan').exists():
            return candidate / 'apps' / 'api'
        if (candidate / 'artisan').exists():
            return candidate
    raise FileNotFoundError('No se encontro apps/api. Ejecuta el cuaderno dentro del repositorio.')

root = find_api_root()
interesting = [
    'routes/web.php',
    'app/Http/Controllers',
    'app/Models',
    'app/Services',
    'app/Http/Requests',
    'resources/js/app',
    'resources/js/layouts',
    'resources/js/features',
    'resources/js/shared',
    'database/migrations',
    'database/seeders',
]

print('Raiz de la API:', root)
for item in interesting:
    path = root / item
    print(f'\n{item}')
    if path.is_dir():
        for child in sorted(path.rglob('*')):
            if child.is_file():
                print('  -', child.relative_to(root))
    elif path.exists():
        print('  -', path.relative_to(root))
    else:
        print('  (no existe)')


## 2. Flujo general de usuario

El flujo previsto para una persona usuaria es:

1. Entrar a `/`. Se muestra siempre la portada publica (`Landing`), con acceso a login y a proyectos.
2. Registrarse (`/register`) o iniciar sesion (`/login`).
3. Ver el listado de proyectos en `/projects`: buscar por titulo, filtrar por favoritos, alternar vista de cuadricula/lista y marcar favoritos.
4. Crear un proyecto en `/projects/create` con nombre, genero, idioma (Espanol o English) y descripcion general. Al guardar vuelve al listado con un mensaje de exito.
5. Entrar al detalle del proyecto en `/projects/{project}`.
6. Definir restricciones creativas: tono emocional, emociones secundarias, tipo de mundo, rol del jugador, nivel de ramificacion, elementos requeridos/prohibidos y notas.
7. Crear piezas de contenido: lore, personajes, escenas y dialogos.
8. Revisar automaticamente advertencias narrativas en el panel de validaciones.
9. Ejecutar analisis emocional local sobre escenas o dialogos.
10. Revisar estimaciones de costo narrativo generadas para escenas y dialogos.

El formulario de edicion (`/projects/{project}/edit`) mantiene todos los campos del proyecto (tipo narrativo, estado, premisa, conflicto central y audiencia).

**Donde se revisa cada cosa desde la app** (rutas relativas a `resources/js/features`):

| Necesidad | Pantalla/ruta | Componente React |
|---|---|---|
| Portada | `/` | `landing/pages/Landing.tsx` |
| Login | `/login` | `auth/pages/Login.tsx` |
| Registro | `/register` | `auth/pages/Register.tsx` |
| Cuenta | `/account` | `account/pages/Edit.tsx` |
| Listado de proyectos | `/projects` | `projects/pages/Index.tsx` + `ProjectCard.tsx` |
| Crear proyecto | `/projects/create` | `projects/pages/Create.tsx` |
| Editar proyecto | `/projects/{project}/edit` | `projects/pages/Edit.tsx` + `ProjectForm.tsx` |
| Ver dashboard del proyecto | `/projects/{project}` | `projects/pages/Show.tsx` |
| Restricciones del proyecto | Dashboard del proyecto | `projects/components/ConstraintForm.tsx` |
| Validaciones narrativas | Dashboard del proyecto | `analysis/components/NarrativeValidationPanel.tsx` |
| Analisis emocional | Dashboard del proyecto | `analysis/components/EmotionalAnalysisPanel.tsx` |
| Lore | `/projects/{project}/lore` | `lore/pages/Index.tsx`, `LoreEntryForm.tsx` |
| Personajes | `/projects/{project}/characters` | `characters/pages/Index.tsx`, `CharacterForm.tsx` |
| Escenas | `/projects/{project}/scenes` | `scenes/pages/Index.tsx`, `SceneForm.tsx` |
| Dialogos | `/projects/{project}/dialogues` | `dialogues/pages/Index.tsx`, `DialogueNodeForm.tsx` |


## 3. Rutas web

Las rutas principales estan en `routes/web.php`.

| Ruta | Metodo | Controlador/accion | Descripcion |
|---|---|---|---|
| `/` | GET | closure | Renderiza la portada publica `Landing` |
| `/login` | GET/POST | `AuthController@create/store` | Mostrar formulario e iniciar sesion |
| `/register` | GET/POST | `AuthController@register/storeRegistration` | Mostrar formulario y crear usuario |
| `/logout` | POST | `AuthController@destroy` | Cerrar sesion |
| `/account` | GET | `AccountController@edit` | Pantalla de cuenta |
| `/account/profile`, `/account/password` | PUT | `AccountController@updateProfile/updatePassword` | Actualizar perfil y contrasena |
| `/projects` | resource | `ProjectController` | CRUD completo de proyectos |
| `/projects/{project}/favorite` | PATCH | `ProjectController@toggleFavorite` | Marcar o quitar un proyecto de favoritos |
| `/projects/{project}/constraints` | POST/PUT | `ProjectConstraintController@store` | Crear o actualizar restricciones |
| `/projects/{project}/lore` | resource sin show | `LoreEntryController` | CRUD de lore dentro de proyecto |
| `/projects/{project}/characters` | resource sin show | `CharacterController` | CRUD de personajes |
| `/projects/{project}/scenes` | resource sin show | `SceneController` | CRUD de escenas |
| `/projects/{project}/dialogues` | resource sin show | `DialogueNodeController` | CRUD de nodos de dialogo |
| `/projects/{project}/emotional-analysis` | GET/POST | `EmotionalAnalysisController` | Redirigir al dashboard o guardar analisis |
| `/projects/{project}/validations` | GET/POST/PATCH | `NarrativeValidationController` | Recalcular validaciones o marcarlas resueltas |
| `/projects/{project}/cost-estimations` | GET/POST | `CostEstimationController` | Redirigir al dashboard o guardar estimacion |

Las rutas de `login` y `register` estan bajo middleware `guest`. Todas las rutas de dominio estan protegidas con middleware `auth` y `verified`. La app marca usuarios registrados como verificados inmediatamente, asi que en la practica el flujo de email verification no esta expuesto al usuario.

Los nombres de pagina que renderizan los controladores (`Projects/Index`, `Auth/Login`, `Landing`...) se traducen a `features/<modulo>/pages/<Pagina>.tsx` en `resources/js/app/resolve-page.ts`.


## 4. Controladores

### `AuthController`

Gestiona login, registro y logout. Valida credenciales, usa `Auth::attempt`, regenera la sesion despues de autenticar y crea usuarios con password hasheado. En registro, tambien asigna `email_verified_at` con la fecha actual y autentica al usuario automaticamente.

### `AccountController`

Muestra la pantalla de cuenta y actualiza el perfil y la contrasena del usuario autenticado (`UpdateAccountProfileRequest`, `UpdateAccountPasswordRequest`).

### `ProjectController`

Controlador principal del dominio. Lista proyectos del usuario con conteos de lore, personajes, escenas y dialogos. Crea, muestra, edita, actualiza y elimina proyectos. Tras crear un proyecto redirige al listado (`projects.index`) con un mensaje de exito. `toggleFavorite` invierte el campo `favorite` y vuelve a la pagina anterior. En `show`, antes de cargar el proyecto, ejecuta `NarrativeValidationService->run($project)`, por lo que entrar al dashboard recalcula advertencias narrativas pendientes. Luego carga relaciones limitadas para el dashboard: restricciones, ultimas entradas de lore, personajes, escenas ordenadas, dialogos con escena/personaje, ultimo analisis emocional, validaciones no resueltas y estimaciones de costo.

### `ProjectConstraintController`

Guarda las restricciones creativas de un proyecto usando `updateOrCreate`. Hay una sola fila de restricciones por proyecto por la columna unica `project_id`.

### `LoreEntryController`

CRUD de entradas de lore. Cada entrada pertenece a un proyecto y se valida que el usuario tenga permiso de actualizar/ver el proyecto. Los metodos `edit`, `update` y `destroy` comprueban que el lore pertenece al proyecto recibido en la ruta.

### `CharacterController`

CRUD de personajes narrativos. Maneja datos de identidad, trasfondo, personalidad, motivacion, miedo, deseo, arco emocional, descripcion visual y notas de voz. Tambien protege que el personaje pertenezca al proyecto correcto.

### `SceneController`

CRUD de escenas. Al crear o actualizar una escena ejecuta `NarrativeCostEstimatorService->createFor($project, $scene)` y guarda el resultado en `cost_estimations`; ademas copia `cost_level` al campo `estimated_cost` de la escena para mostrarlo rapido en UI.

### `DialogueNodeController`

CRUD de nodos de dialogo. Al crear/editar carga listas de escenas, personajes y otros nodos para formularios. Al guardar sincroniza opciones: elimina las opciones anteriores y crea las nuevas. Tambien ejecuta la estimacion de costo sobre el nodo de dialogo.

### `EmotionalAnalysisController`

Permite elegir una fuente `scene` o `dialogue`, resolverla dentro del proyecto y crear un analisis emocional local. Para escenas usa `summary`; para dialogos usa `text`. Si no hay texto, aborta con codigo 422. El mensaje de exito dice explicitamente `Analisis emocional simulado guardado`.

### `NarrativeValidationController`

Permite recalcular validaciones y marcar una validacion como resuelta. Al recalcular, delega en `NarrativeValidationService`. Al resolver, actualiza `resolved` a `true`.

### `CostEstimationController`

Permite crear estimaciones de costo manualmente para escenas o dialogos desde una fuente seleccionada. Para escenas, tambien actualiza `estimated_cost` en la tabla `scenes`.


## 5. Modelos y relaciones

| Modelo | Tabla | Relaciones principales | Comentario |
|---|---|---|---|
| `User` | `users` | tiene muchos `Project` | Usuario propietario de proyectos |
| `Project` | `projects` | pertenece a `User`; tiene `constraint`, `loreEntries`, `characters`, `scenes`, `dialogueNodes`, `emotionalAnalyses`, `narrativeValidations`, `costEstimations`, `aiGenerations` | Nodo central de la app; `favorite` se castea a booleano |
| `ProjectConstraint` | `project_constraints` | pertenece a `Project` | Restricciones creativas del proyecto |
| `LoreEntry` | `lore_entries` | pertenece a `Project`; tiene analisis emocionales polimorficos declarados | Piezas de mundo, historia, facciones, lugares, objetos, eventos o conceptos |
| `Character` | `characters` | pertenece a `Project`; tiene muchos `DialogueNode` | Personajes narrativos |
| `Scene` | `scenes` | pertenece a `Project`; tiene muchos `DialogueNode`; tiene muchos analisis/costos polimorficos | Escenas ordenadas del proyecto |
| `DialogueNode` | `dialogue_nodes` | pertenece a `Project`, `Scene`, `Character`; tiene `options`; tiene `incomingOptions`; tiene analisis/costos polimorficos | Unidad de dialogo, puede enlazarse con opciones |
| `DialogueOption` | `dialogue_options` | pertenece a `DialogueNode`; puede apuntar a `nextDialogueNode` | Opcion del jugador y salto opcional |
| `EmotionalAnalysis` | `emotional_analyses` | pertenece a `Project`; `morphTo analyzable` | Resultado de analisis emocional local |
| `NarrativeValidation` | `narrative_validations` | pertenece a `Project` | Advertencias narrativas pendientes o resueltas |
| `CostEstimation` | `cost_estimations` | pertenece a `Project`; `morphTo costable` | Estimacion de costo narrativo para escena/dialogo |
| `AiGeneration` | `ai_generations` | pertenece a `Project` y `User` | Tabla preparada para guardar generaciones IA, pero no se usa actualmente desde controladores/servicios |

Los modelos usan `$fillable` para controlar asignacion masiva. Algunos campos JSON se castean a arrays: emociones secundarias, elementos requeridos/prohibidos, puntajes emocionales, sugerencias y factores de costo.


## 6. Base de datos

El motor configurado en `.env` es MySQL (base `gamerdust`). Las migraciones de dominio son:

- `database/migrations/2026_06_04_000000_create_gamerdust_tables.php`: crea todas las tablas del dominio.
- `database/migrations/2026_10_06_000000_add_language_and_favorite_to_projects.php`: anade `language` y `favorite` a `projects`.

### `projects`

Representa un proyecto narrativo del usuario. Campos clave: `user_id`, `title`, `description`, `game_genre`, `language` (por defecto `Español`), `narrative_type`, `premise`, `central_conflict`, `target_audience`, `status`, `favorite` (booleano, por defecto falso), timestamps. `status` tiene indice. Si se borra el usuario, se borran sus proyectos por `cascadeOnDelete`.

### `project_constraints`

Una fila por proyecto gracias a `project_id` unico. Guarda tono emocional, emocion dominante, emociones secundarias JSON, tipo de mundo, rol del jugador, ramificacion, elementos requeridos/prohibidos JSON y notas creativas.

### `lore_entries`

Guarda piezas de lore. Campos: `project_id`, `title`, `type`, `description`, `narrative_function`, `emotional_tone`. `type` tiene indice.

### `characters`

Guarda personajes. Campos: nombre, rol, arquetipo, edad, genero, trasfondo, personalidad, estilo de habla, motivacion, miedo, deseo, arco emocional, descripcion visual y notas de voz.

### `scenes`

Guarda escenas. Campos: titulo, orden, resumen, conflicto, emocion esperada, nivel de interes, tipo, decision del jugador, consecuencia y costo estimado. Tiene indice compuesto `project_id` + `order` para ordenar escenas de un proyecto.

### `dialogue_nodes`

Guarda nodos de dialogo. Puede asociarse a escena y personaje. `node_key` es unico dentro de cada proyecto con indice unico `project_id` + `node_key`. Incluye tipo de dialogo, texto, tono emocional y condicion de disparo.

### `dialogue_options`

Guarda opciones del jugador para un nodo. Cada opcion puede tener texto, consecuencia, condicion requerida y un salto a otro nodo por `next_dialogue_node_id`.

### `emotional_analyses`

Guarda resultados de analisis emocional. Usa relacion polimorfica `analyzable`, asi que puede apuntar a distintos tipos de contenido. En la practica actual el controlador permite analizar escenas y dialogos. Guarda texto original, emocion dominante, valence/arousal/dominance, puntajes JSON, interpretacion y sugerencias JSON.

### `narrative_validations`

Guarda advertencias de coherencia narrativa. Campos: tipo de validacion, severidad, mensaje, recomendacion y `resolved`. Tiene indices para tipo, severidad y resuelto.

### `cost_estimations`

Guarda costo narrativo calculado. Usa relacion polimorfica `costable`, actualmente para escenas y dialogos. Incluye nivel (`low`, `medium`, `high`), puntaje numerico, factores JSON y explicacion.

### `ai_generations`

Tabla preparada para guardar generaciones de IA: proyecto, usuario, tipo de generacion, prompt, response JSON y modelo. Actualmente no hay controlador ni servicio que la escriba.

### Tablas de infraestructura de Laravel

`users`, `password_reset_tokens`, `sessions` (las sesiones usan el driver `database`), `cache`, `cache_locks`, `jobs`, `job_batches` y `failed_jobs`.


In [ ]:
# Celda opcional: inspeccionar un SQLite local si apps/api/database/database.sqlite existe.
# El proyecto usa MySQL (ver apps/api/.env), asi que normalmente solo avisara que no hay SQLite.
# Requiere haber ejecutado antes la celda de inventario (define find_api_root).
import sqlite3

api_root = find_api_root()
db_path = api_root / 'database' / 'database.sqlite'
if not db_path.exists():
    print('No existe database/database.sqlite. El motor configurado en apps/api/.env es MySQL.')
else:
    con = sqlite3.connect(db_path)
    cur = con.cursor()
    tables = [row[0] for row in cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")]
    for table in tables:
        print(f'\n[{table}]')
        for col in cur.execute(f'PRAGMA table_info({table})'):
            print(col)
    con.close()


## 7. Validaciones de entrada

La app usa `FormRequest` para reglas de datos.

| Request | Uso | Reglas destacadas |
|---|---|---|
| `StoreProjectRequest` | Crear proyecto | `title` obligatorio; `language` obligatorio y debe ser `Español` o `English`; `narrative_type` y `status` son opcionales (`sometimes`) y la base aplica `balanced_narrative` y `draft` por defecto |
| `UpdateProjectRequest` | Actualizar proyecto | `narrative_type` y `status` siguen siendo obligatorios; autoriza con policy sobre el proyecto |
| `StoreProjectConstraintRequest` | Guardar restricciones | Convierte strings separados por comas/saltos en arrays para emociones y elementos; valida arrays y textos |
| `StoreLoreEntryRequest` | Crear/editar lore | `title` requerido; `type` debe ser `rule`, `history`, `faction`, `location`, `object`, `event` o `concept` |
| `StoreCharacterRequest` | Crear/editar personaje | `name` requerido; edad entera entre 0 y 200; textos opcionales para motivacion, miedo, deseo, etc. |
| `StoreSceneRequest` | Crear/editar escena | `title` requerido; `order` entero minimo 1; `interest_level` entre 1 y 10 |
| `StoreDialogueNodeRequest` | Crear/editar dialogo | `node_key` requerido y unico por proyecto; `scene_id`/`character_id` deben pertenecer al proyecto; `dialogue_type` limitado a tipos declarados; opciones opcionales con salto a nodo valido |
| `UpdateAccountProfileRequest`, `UpdateAccountPasswordRequest` | Cuenta | Validan nombre/email y cambio de contrasena |

La autorizacion se apoya en `ProjectPolicy`: un usuario solo puede ver, actualizar o eliminar proyectos que le pertenecen.


## 8. Servicios de dominio

### `NarrativeValidationService`

Ejecuta reglas simples de coherencia. Antes de recalcular, borra las validaciones no resueltas del proyecto para no duplicarlas. Luego crea hallazgos si detecta:

- Proyecto sin premisa: severidad `high`.
- Personaje sin motivacion: severidad `medium`.
- Escena sin conflicto: severidad `medium`.
- Lore sin funcion narrativa: severidad `low`.

Estas advertencias se revisan en el dashboard del proyecto, dentro de `NarrativeValidationPanel.tsx`. El usuario puede pulsar `Recalcular` o `Resolver`.

### `EmotionalAnalysisService`

No llama a un proveedor de IA. Normaliza el texto a minusculas y puntua emociones mediante palabras clave en ingles y espanol. Emociones consideradas: `joy`, `fear`, `anger`, `sadness`, `tension`. La emocion dominante es la de mayor puntaje. Tambien calcula valores simulados de `valence`, `arousal` y `dominance`, y guarda sugerencias genericas.

Palabras clave de ejemplo:

- `joy`: hope, joy, victory, friend, celebration, luz.
- `fear`: fear, terror, dark, threat, monster, miedo.
- `anger`: rage, betrayal, revenge, war, furia, ira.
- `sadness`: loss, grief, alone, death, tristeza, duelo.
- `tension`: conflict, choice, risk, secret, conflicto, riesgo.

### `NarrativeCostEstimatorService`

Calcula costo narrativo con reglas locales. Para escenas, sube el puntaje si hay decision del jugador, consecuencia y cuatro o mas dialogos asociados. Para dialogos, parte de costo bajo si el tipo es `bark` o `ambient`; otros tipos arrancan en medio. Si hay mas de una opcion, suma ramificacion. El resultado se guarda en `cost_estimations` y, para escenas, tambien en `scenes.estimated_cost`.

## 9. IA: estado actual

**Conclusion:** la aplicacion todavia no tiene IA real integrada.

Evidencias en el codigo:

- `EmotionalAnalysisController` devuelve el mensaje `Analisis emocional simulado guardado`.
- `EmotionalAnalysisPanel.tsx` indica que es un `Placeholder local con palabras clave, listo para conectar IA despues`.
- `EmotionalAnalysisService` usa `Str::contains` y listas de palabras clave; no usa OpenAI, Anthropic, Hugging Face, Ollama ni ningun cliente HTTP.
- `NarrativeValidationService` usa reglas deterministas simples, no IA.
- `NarrativeCostEstimatorService` usa heuristicas deterministas, no IA.
- Existe el modelo/tabla `AiGeneration`, pero actualmente no se observa uso desde rutas, controladores o servicios.

Por tanto, la app esta preparada conceptualmente para funciones asistidas por IA, especialmente por la tabla `ai_generations`, pero hasta ahora lo que hay es simulacion local y reglas de negocio.

## 10. Frontend: paginas y componentes

El frontend esta en `resources/js` y se organiza por funcionalidades.

```
resources/js/
├── app.tsx                 Entrada de Vite: inicializa Inertia y monta React
├── app/resolve-page.ts     Traduce "Projects/Index" -> features/projects/pages/Index.tsx
├── layouts/
│   ├── SiteLayout.tsx      Cabecera + contenido + pie (paginas publicas y listado/creacion de proyectos)
│   ├── ProjectLayout.tsx   Carcasa de paginas internas (antes AppShell): cabecera, avisos flash y pie
│   └── components/         Brand, Header, Footer
├── features/
│   ├── landing/            pages/Landing.tsx
│   ├── auth/               pages/Login.tsx, Register.tsx
│   ├── account/            pages/Edit.tsx
│   ├── projects/           pages, components (ProjectCard, ProjectForm, ProjectStats, ConstraintForm), projects.types.ts
│   ├── characters/         pages, components, characters.types.ts
│   ├── scenes/             pages, components, scenes.types.ts
│   ├── lore/               pages, components, lore.types.ts
│   ├── dialogues/          pages, components, dialogues.types.ts
│   └── analysis/           components (EmotionalAnalysisPanel, NarrativeValidationPanel, CostEstimationBadge) y tipos de emocion, costo y validacion
└── shared/lib/styles.ts    Clases comunes (button, field), lista de generos e imagen por genero
```

### Entrada React/Inertia

`app.tsx` inicializa Inertia, resuelve paginas con `resolvePage` (carga `features/*/pages/**/*.tsx`), monta React con `createRoot` y configura la barra de progreso color amber. Los controladores siguen usando los mismos nombres de pagina; `resolve-page.ts` pasa la primera parte a minusculas para obtener el modulo (`Lore/Index` -> `features/lore/pages/Index.tsx`) y `Landing` a `features/landing/pages/Landing.tsx`.

### Layouts

- `SiteLayout.tsx`: cabecera, contenido y pie con el diseno nuevo.
- `ProjectLayout.tsx`: usado por las paginas internas (lore, personajes, escenas, dialogos, detalle/edicion de proyecto y cuenta). Reutiliza `Header` y `Footer`, y muestra los mensajes flash de exito/error.
- `Header.tsx`: navegacion, nombre del usuario y boton de salir si hay sesion; enlace "Comenzar a escribir" si no la hay.

### Paginas de proyectos

- `projects/pages/Index.tsx`: lista proyectos paginados con busqueda por titulo, pestanas Todos/Favoritos, vista cuadricula/lista, tarjeta para crear uno nuevo y paginacion.
- `projects/pages/Create.tsx`: formulario de nombre, genero, idioma y descripcion (usa `useForm` de Inertia y envia a `POST /projects`).
- `projects/pages/Edit.tsx`: muestra `ProjectForm` para editar todos los campos.
- `projects/pages/Show.tsx`: dashboard central del proyecto. Muestra resumen, estadisticas, restricciones, validaciones, analisis emocional, accesos rapidos y secciones recientes de lore, personajes, escenas y dialogos.

### Otras paginas

- `landing/pages/Landing.tsx`: portada con hero, herramientas y llamada a la accion.
- `auth/pages/Login.tsx` y `Register.tsx`: formularios de acceso y registro.
- `account/pages/Edit.tsx`: perfil y contrasena.
- `lore`, `characters`, `scenes`, `dialogues`: cada modulo tiene `Index`, `Create` y `Edit`.

### Componentes de dominio

| Componente | Funcion |
|---|---|
| `ProjectForm` | Formulario de edicion de proyecto: titulo, descripcion, genero, tipo narrativo, estado, premisa, conflicto, audiencia |
| `ProjectCard` | Tarjeta de proyecto: imagen segun genero, favorito (`PATCH /projects/{id}/favorite`), numero de escenas, editar, eliminar y abrir |
| `ProjectStats` | Muestra conteos de lore, personajes, escenas y dialogos |
| `ConstraintForm` | Edita restricciones creativas del proyecto |
| `NarrativeValidationPanel` | Lista advertencias, permite recalcular y resolver |
| `EmotionalAnalysisPanel` | Permite elegir escena/dialogo y ejecutar analisis emocional local |
| `CostEstimationBadge` | Muestra etiqueta visual de costo bajo/medio/alto |
| `LoreEntryForm` / `LoreEntryCard` | Formulario y tarjeta de lore |
| `CharacterForm` / `CharacterCard` | Formulario y tarjeta de personaje |
| `SceneForm` / `SceneCard` | Formulario y tarjeta de escena, incluye costo estimado si aplica |
| `DialogueNodeForm` / `DialogueNodeCard` | Formulario de nodo con opciones del jugador y salto a otro nodo; tarjeta con escena/personaje/opciones |

Los formularios usan `useForm` de Inertia. Eso significa que mantienen estado local en React, envian `post` o `put`, reciben errores de validacion desde Laravel y muestran feedback sin construir manualmente una API REST.

**Pendiente de la migracion visual:** las paginas internas de lore, personajes, escenas, dialogos, detalle de proyecto, cuenta, login y registro siguen con el estilo anterior zinc/ambar.


## 11. Seed inicial

`database/seeders/DatabaseSeeder.php` crea datos de ejemplo para probar el flujo completo:

- Usuario: `test@example.com` con password `password`.
- Proyecto: `Dustfall Protocol`.
- Restricciones: tono melancolico esperanzado, emocion dominante tension, mundo de colonia aislada, rol archivista, ramificacion moderada.
- Lore: `El Polvo de Vigilia`.
- Personaje: `Mara Venn`.
- Escena: `Archivo bajo ceniza`.
- Dialogo: `archive_intro_001` con una opcion de jugador.

Este seed permite entrar, revisar el dashboard y probar analisis, validaciones y costos sin crear datos desde cero.

## 12. Seguridad y autorizacion

La seguridad actual se basa en:

- Middleware `auth` y `verified` para rutas de dominio.
- `ProjectPolicy` para controlar acceso por propietario.
- Comprobaciones manuales `abort_unless(..., 404)` en recursos hijos para evitar acceder a personajes, escenas, lore o dialogos de otro proyecto.
- Validaciones con `FormRequest` para entradas persistentes.
- `Hash::make` para passwords.
- Regeneracion de sesion tras login y registro.

Punto a vigilar: como los usuarios se marcan verificados automaticamente al registrarse, `verified` no representa una verificacion real por email en este estado.

## 13. Ciclos principales de datos

### Crear proyecto

`projects/pages/Create.tsx` envia nombre, genero, idioma y descripcion a `ProjectController@store`. `StoreProjectRequest` valida; `narrative_type` y `status` toman sus valores por defecto de la base. El proyecto se crea desde `$request->user()->projects()`. Laravel redirige al listado con el mensaje "Proyecto narrativo creado".

### Marcar favorito

`ProjectCard` envia `PATCH /projects/{project}/favorite`. `ProjectController@toggleFavorite` autoriza la actualizacion, invierte `favorite` y vuelve a la pagina anterior. La busqueda y el filtro de favoritos del listado se aplican en el navegador sobre la pagina cargada.

### Ver dashboard

`ProjectController@show` autoriza el proyecto, recalcula validaciones narrativas y carga relaciones resumidas. `projects/pages/Show.tsx` reparte la informacion a paneles y tarjetas.

### Crear escena

`SceneForm` envia a `SceneController@store`. Se crea la escena, se calcula costo con `NarrativeCostEstimatorService`, se guarda una fila en `cost_estimations` y se actualiza `scenes.estimated_cost`.

### Crear dialogo

`DialogueNodeForm` envia nodo y opciones. `DialogueNodeController` crea el nodo, borra/sincroniza opciones, calcula costo y redirige al listado de dialogos.

### Analizar emocion

`EmotionalAnalysisPanel` envia `source_type` y `source_id`. El controlador resuelve la escena/dialogo dentro del proyecto, toma el texto, ejecuta `EmotionalAnalysisService` y guarda `emotional_analyses`.

### Validar narrativa

El panel de validaciones ejecuta POST a `/projects/{project}/validations`. El servicio borra validaciones no resueltas y recrea advertencias segun reglas simples. Resolver una advertencia hace PATCH y marca `resolved = true`.


In [ ]:
# Celda opcional: buscar referencias a IA/generacion/modelos externos en el codigo.
# Requiere haber ejecutado antes la celda de inventario (define find_api_root).
import re

terms = re.compile(r'OpenAI|Anthropic|Hugging|Ollama|ai_generation|AiGeneration|model|prompt|response|simulado|placeholder', re.IGNORECASE)
api_root = find_api_root()
skip = {'.git', 'node_modules', 'vendor', 'storage', 'build'}

for path in api_root.rglob('*'):
    if not path.is_file():
        continue
    if any(part in skip for part in path.parts):
        continue
    if path.suffix.lower() not in {'.php', '.tsx', '.ts', '.js', '.json'}:
        continue
    text = path.read_text(encoding='utf-8', errors='ignore')
    if terms.search(text):
        print(path.relative_to(api_root))

# El motor de analisis (apps/narrative-engine) es solo un esqueleto de archivos vacios por ahora.
engine = api_root.parent / 'narrative-engine'
if engine.exists():
    files = [p for p in engine.rglob('*') if p.is_file()]
    empty = [p for p in files if p.stat().st_size == 0]
    print(f'\nnarrative-engine: {len(files)} archivos, {len(empty)} vacios')


## 14. Como correr y revisar la app

Requisitos: PHP 8.4 o superior (con la extension `zip` habilitada para Composer), Composer, Node 22+ y MySQL. En Laragon, selecciona PHP 8.4+ y apunta el host virtual a `apps/api/public`.

Comandos utiles desde `apps/api`:

```bash
composer install
npm install
php artisan key:generate
php artisan migrate --seed
composer run dev
```

Alternativa en dos terminales: `php artisan serve` y `npm run dev`. Abre la URL de Laravel (por ejemplo `http://localhost:8000` o `http://gamerdustv1.test`), no el puerto 5173 de Vite, que solo sirve los assets.

Desde VS Code estan definidas las tareas `api: serve`, `web: vite dev`, `dev: api + web` y `api: tests` en `.vscode/tasks.json`.

Con el seed, se puede entrar con:

- Email: `test@example.com`
- Password: `password`

Para pruebas automatizadas, `composer test` ejecuta `php artisan test`. En el estado revisado solo aparecen tests de ejemplo (`tests/Feature/ExampleTest.php`, `tests/Unit/ExampleTest.php`), por lo que todavia falta cobertura especifica del dominio Gamerdust. Las pruebas de recorridos completos iran en `tests/e2e/` (vacio por ahora).


## 15. Estado actual y siguientes pasos sugeridos

La app ya tiene una base funcional para gestionar narrativa de videojuegos: autenticacion, proyectos (con genero, idioma y favoritos), restricciones, lore, personajes, escenas, dialogos, validaciones, analisis emocional simulado y costos narrativos, ademas de una portada publica y un listado de proyectos con el nuevo diseno.

Lo que todavia no tiene:

- IA real conectada a un modelo externo.
- Uso efectivo de la tabla `ai_generations`.
- Implementacion del motor `apps/narrative-engine` (solo esqueleto).
- API dedicada para integraciones externas.
- Tests de dominio para controladores, policies, requests y servicios.
- Validaciones narrativas avanzadas con contexto cruzado profundo.
- Migracion visual completa: varias paginas internas siguen con el estilo anterior.
- Piezas del C4 aun sin crear en el frontend: `router.tsx`, `providers.tsx`, capas `*.api.ts` y `*.validation.ts`, internacionalizacion (`i18n/`) y paginas de recuperacion de contrasena.

Posibles siguientes pasos tecnicos:

1. Definir la IA a usar y conectarla a traves de `apps/narrative-engine`, guardando prompts/respuestas en `ai_generations`.
2. Ampliar `NarrativeValidationService` con reglas de continuidad, contradicciones, arcos emocionales y dependencia entre escenas.
3. Agregar tests para ownership de proyectos y recursos hijos.
4. Crear tests unitarios de los servicios de costo, validacion y analisis emocional.
5. Aplicar el nuevo diseno al resto de paginas internas y a login/registro.
6. Mejorar UI de grafo de dialogos para visualizar saltos entre nodos.
7. Agregar filtros/busqueda en lore, personajes, escenas y dialogos, e i18n Espanol/English.
8. Decidir si el backend migra a NestJS/PostgreSQL como indica el C4 o se mantiene Laravel/MySQL.
